# Phase 1 — Data Understanding
## Rappi Delivery Operations Dataset

**Purpose:** Confirm sheet roles, key fields, and expected grain before cleaning.

The workbook contains hourly delivery operations for Monterrey across 30 days and 14 zones, covering supply, demand, earnings, and weather.

**Expected grain:** one `COUNTRY × DATE × HOUR × CITY × ZONE` observation. Notebook 01 validates it formally.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import DATA_PATH
from src.io_utils import load_all_sheets

print("Libraries loaded.")


Libraries loaded.


---
## A. Sheet Discovery

Load all sheets and inspect their structure.


In [2]:
# Step 1: load ALL sheets — no names assumed
sheets = load_all_sheets(DATA_PATH)

print(f"Workbook: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Sheets found ({len(sheets)}): {list(sheets.keys())}")


Workbook: data\rappi_delivery_case_data.xlsx
Sheets found (3): ['RAW_DATA', 'ZONE_INFO', 'ZONE_POLYGONS']


In [3]:
# Step 2: shape, columns, and dtypes for every sheet
for name, df in sheets.items():
    print(f"\n{'='*60}")
    print(f"SHEET : {name}")
    print(f"Shape : {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
    print(f"Columns ({df.shape[1]}): {list(df.columns)}")
    print("\nData types:")
    print(df.dtypes.to_string())



SHEET : RAW_DATA
Shape : 10,080 rows  ×  9 columns
Columns (9): ['COUNTRY', 'DATE', 'HOUR', 'CITY', 'ZONE', 'CONNECTED_RT', 'ORDERS', 'EARNINGS', 'PRECIPITATION_MM']

Data types:
COUNTRY                 str
DATE                    str
HOUR                  int64
CITY                    str
ZONE                    str
CONNECTED_RT          int64
ORDERS                int64
EARNINGS            float64
PRECIPITATION_MM    float64

SHEET : ZONE_INFO
Shape : 14 rows  ×  4 columns
Columns (4): ['ZONE', 'LATITUDE_CENTER', 'LONGITUDE_CENTER', 'DESCRIPTION']

Data types:
ZONE                    str
LATITUDE_CENTER     float64
LONGITUDE_CENTER    float64
DESCRIPTION             str

SHEET : ZONE_POLYGONS
Shape : 14 rows  ×  3 columns
Columns (3): ['ZONE_ID', 'ZONE_NAME', 'GEOMETRY_WKT']

Data types:
ZONE_ID         int64
ZONE_NAME         str
GEOMETRY_WKT      str


In [4]:
# Step 3: sample rows for every sheet — see what the data actually looks like
for name, df in sheets.items():
    print(f"\n{'='*60}")
    print(f"SHEET : {name}  — first 5 rows")
    print(df.head(5).to_string())



SHEET : RAW_DATA  — first 5 rows
  COUNTRY        DATE  HOUR       CITY                ZONE  CONNECTED_RT  ORDERS  EARNINGS  PRECIPITATION_MM
0  Mexico  2024-03-01     0  Monterrey              Centro             6       2      54.4               0.0
1  Mexico  2024-03-01     0  Monterrey       Mitras Centro             5       1      52.4               0.0
2  Mexico  2024-03-01     0  Monterrey      Apodaca Centro             4       1      59.1               0.0
3  Mexico  2024-03-01     0  Monterrey            Escobedo             3       1      53.3               0.0
4  Mexico  2024-03-01     0  Monterrey  Carretera Nacional             2       0      55.0               0.0

SHEET : ZONE_INFO  — first 5 rows
                 ZONE  LATITUDE_CENTER  LONGITUDE_CENTER                                               DESCRIPTION
0              Centro          25.6823         -100.3185  Downtown Monterrey – high-density commercial/residential
1       Mitras Centro          25.7240         

In [5]:
# Step 4: numeric summary for every sheet — ranges, nulls, cardinality
for name, df in sheets.items():
    print(f"\n{'='*60}")
    print(f"SHEET : {name}  — null counts")
    print(df.isnull().sum().to_string())
    print(f"\nUnique values per column:")
    print(df.nunique().to_string())
    numeric_cols = df.select_dtypes(include="number").columns
    if len(numeric_cols):
        print(f"\nNumeric describe:")
        print(df[numeric_cols].describe().round(2).to_string())



SHEET : RAW_DATA  — null counts
COUNTRY             0
DATE                0
HOUR                0
CITY                0
ZONE                0
CONNECTED_RT        0
ORDERS              0
EARNINGS            0
PRECIPITATION_MM    0

Unique values per column:
COUNTRY               1
DATE                 30
HOUR                 24
CITY                  1
ZONE                 14
CONNECTED_RT         45
ORDERS               56
EARNINGS            352
PRECIPITATION_MM     35

Numeric describe:
           HOUR  CONNECTED_RT    ORDERS  EARNINGS  PRECIPITATION_MM
count  10080.00      10080.00  10080.00  10080.00          10080.00
mean      11.50          9.14      9.44     57.70              0.25
std        6.92          7.65     10.61      8.29              1.27
min        0.00          1.00      0.00     49.00              0.00
25%        5.75          3.00      1.00     52.40              0.00
50%       11.50          7.00      6.00     55.60              0.00
75%       17.25         13.00  

---
### Sheet role assignment

After running the cells above we can see what each sheet contains. Now we name them explicitly based on what we actually observed.


In [6]:
# Step 5: assign sheet roles based on what we saw above
# ── Edit these three lines to match the actual sheet names printed in Step 1 ──

sheet_names = list(sheets.keys())
print("Available sheet names:", sheet_names)

raw_sheet_name       = sheet_names[0]   # hourly operational fact table
zone_info_sheet_name = sheet_names[1]   # zone dimension with centroid coordinates
polygons_sheet_name  = sheet_names[2]   # zone polygon geometries (WKT)

raw       = sheets[raw_sheet_name]
zone_info = sheets[zone_info_sheet_name]
polygons  = sheets[polygons_sheet_name]

print(f"\nAssigned:")
print(f"  raw       ← '{raw_sheet_name}'       {raw.shape}")
print(f"  zone_info ← '{zone_info_sheet_name}'     {zone_info.shape}")
print(f"  polygons  ← '{polygons_sheet_name}'  {polygons.shape}")


Available sheet names: ['RAW_DATA', 'ZONE_INFO', 'ZONE_POLYGONS']

Assigned:
  raw       ← 'RAW_DATA'       (10080, 9)
  zone_info ← 'ZONE_INFO'     (14, 4)
  polygons  ← 'ZONE_POLYGONS'  (14, 3)


---
## B. Sheet Inventory Summary

Quick reference of what we found.


In [7]:
# Build inventory from what we actually discovered
rows = []
for name, df in sheets.items():
    role = {raw_sheet_name: "fact table", zone_info_sheet_name: "dimension / lookup",
            polygons_sheet_name: "geospatial reference"}.get(name, "unknown")
    rows.append({
        "Sheet": name,
        "Role": role,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Column names": ", ".join(df.columns.tolist()),
    })

pd.DataFrame(rows)


,Sheet,Role,Rows,Columns,Column names
0,RAW_DATA,fact table,10080,9,"COUNTRY, DATE, HOUR, CITY, ZONE, CONNECTED_RT,..."
1,ZONE_INFO,dimension / lookup,14,4,"ZONE, LATITUDE_CENTER, LONGITUDE_CENTER, DESCR..."
2,ZONE_POLYGONS,geospatial reference,14,3,"ZONE_ID, ZONE_NAME, GEOMETRY_WKT"


---
## C. Variable Dictionary

After inspecting the columns and sample values above, we document every variable's meaning, type, and known risks.


---
## E. Initial Merge Strategy

### Merge 1: `RAW_DATA` + `ZONE_INFO`
| Attribute | Value |
|---|---|
| Left table | `RAW_DATA` |
| Right table | `ZONE_INFO` |
| Join key | `ZONE` (both tables) |
| Join type | Left join |
| Expected cardinality | Many-to-one |
| Risk | Name mismatches (whitespace, casing, accents) |

### Merge 2: Enriched data + `ZONE_POLYGONS`
| Attribute | Value |
|---|---|
| Left table | `RAW_DATA` (or enriched version) |
| Right table | `ZONE_POLYGONS` |
| Preferred key | `ZONE_ID` if present in raw data; otherwise `ZONE` ↔ `ZONE_NAME` |
| Practical key | Zone name with normalisation (ZONE_CLEAN ↔ ZONE_NAME_CLEAN) |
| Join type | Left join |
| Expected cardinality | Many-to-one |
| Risk | Name-based join requires strong validation before execution |

### Key pre-merge validation steps (performed in Notebook 01)
1. Verify `ZONE` is unique in `ZONE_INFO`
2. Verify `ZONE_NAME` is unique in `ZONE_POLYGONS`
3. Compare zone sets across all three sheets
4. Standardise zone name keys (strip, upper, remove accents)
5. Check merge cardinality before executing any join

In [8]:
# Quick peek at actual column names and first values to validate variable dictionary
print("=== RAW_DATA columns ===")
print(list(raw.columns))
print("\n=== ZONE_INFO columns ===")
print(list(zone_info.columns))
print("\n=== ZONE_POLYGONS columns ===")
print(list(polygons.columns))

# Show ZONE values in each sheet
print("\n--- Unique ZONE values in RAW_DATA ---")
if "ZONE" in raw.columns:
    print(sorted(raw["ZONE"].unique().tolist()))

print("\n--- ZONE_INFO zones ---")
if "ZONE" in zone_info.columns:
    print(sorted(zone_info["ZONE"].unique().tolist()))

print("\n--- ZONE_POLYGONS zone names ---")
name_col = "ZONE_NAME" if "ZONE_NAME" in polygons.columns else polygons.columns[1]
print(sorted(polygons[name_col].unique().tolist()))

=== RAW_DATA columns ===
['COUNTRY', 'DATE', 'HOUR', 'CITY', 'ZONE', 'CONNECTED_RT', 'ORDERS', 'EARNINGS', 'PRECIPITATION_MM']

=== ZONE_INFO columns ===
['ZONE', 'LATITUDE_CENTER', 'LONGITUDE_CENTER', 'DESCRIPTION']

=== ZONE_POLYGONS columns ===
['ZONE_ID', 'ZONE_NAME', 'GEOMETRY_WKT']

--- Unique ZONE values in RAW_DATA ---
['Apodaca Centro', 'Carretera Nacional', 'Centro', 'Cumbres Poniente', 'Escobedo', 'Independencia', 'La Fe', 'MTY_Apodaca_Huinalá', 'MTY_Guadalupe', 'Mitras Centro', 'San Nicolás', 'San Pedro', 'Santa Catarina', 'Santiago']

--- ZONE_INFO zones ---
['Apodaca Centro', 'Carretera Nacional', 'Centro', 'Cumbres Poniente', 'Escobedo', 'Independencia', 'La Fe', 'MTY_Apodaca_Huinalá', 'MTY_Guadalupe', 'Mitras Centro', 'San Nicolás', 'San Pedro', 'Santa Catarina', 'Santiago']

--- ZONE_POLYGONS zone names ---
['Apodaca Centro', 'Carretera Nacional', 'Centro', 'Cumbres Poniente', 'Escobedo', 'Independencia', 'La Fe', 'MTY_Apodaca_Huinalá', 'MTY_Guadalupe', 'Mitras Centro'

---
## F. Notebook 00 — Summary

| Section | Outcome |
|---|---|
| Sheet inventory | 3 sheets loaded: 1 fact table, 1 dimension, 1 geospatial reference |
| Proposed table grain | `COUNTRY` × `DATE` × `HOUR` × `CITY` × `ZONE` (to be validated in NB 01) |
| Variable dictionary | 15 variables documented across 3 sheets |
| Data model | Star-like: RAW_DATA ← ZONE_INFO (via ZONE), RAW_DATA ← ZONE_POLYGONS (via zone name) |
| Merge strategy | Left joins, name-based key with normalisation required |

### Key risks to address in Notebook 01
1. **Name mismatches** — zone names may differ in whitespace, casing, or accents across sheets
2. **Grain duplicates** — if the expected grain is not unique, aggregation decisions must be made
3. **ZONE_ID absent in RAW_DATA** — polygon joins must rely on name normalisation
4. **EARNINGS ambiguity** — exact definition (gross/net, per-order vs total) needs confirmation
5. **PRECIPITATION_MM sparsity** — most rows will be 0; tail events carry the signal

> **Next step:** Run `01_data_loading_quality_cleaning.ipynb` to standardise types, validate grain, and produce clean datasets.